# Async Operations in Agentic AI Systems — Hands-On Notebook

**Companion notebook** to the tutorial doc. Every cell is runnable on **Python 3.11+**.

**What you will build up progressively:**
1. Baseline: `asyncio` primitives (event loop, coroutine, Task, gather)
2. Fake LLM + fake tools to simulate an agent workload without spending API credits
3. Sync-vs-async wall-time comparison for a 5-step agent
4. Parallel tool fan-out with `gather` and `TaskGroup`
5. Streaming token generation with `async for`
6. Concurrency control: `Semaphore`, timeout, retry with backoff, cancellation
7. Multi-agent supervisor dispatching work concurrently
8. Background fire-and-forget with `create_task`
9. Real-provider snippets: `AsyncOpenAI`, `AsyncAnthropic`, Databricks async patterns (read-only, gated)
10. Load harness: how many concurrent agent runs before latency degrades?

> **Rule for the entire notebook:** anything that would block the event loop (`time.sleep`, `requests.get`, sync DB drivers) is either replaced with an `asyncio` equivalent or wrapped with `asyncio.to_thread`. Break that rule and your 'async' agent silently runs single-threaded.

## 0. Environment check

You need Python 3.11+ for `asyncio.TaskGroup` and `asyncio.timeout`. If you're on 3.10 or older, the fan-out and structured-concurrency cells will fail — upgrade the kernel.

In [ ]:
import sys, platform
assert sys.version_info >= (3, 11), f"Need Python 3.11+, got {sys.version}"
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

In [ ]:
# Optional installs — skip any you don't need
# !pip install --quiet httpx tenacity nest_asyncio openai anthropic

In [ ]:
# Jupyter already runs an event loop, so we patch it to allow nested asyncio.run.
# In a plain .py script this line is NOT needed.
import nest_asyncio; nest_asyncio.apply()

import asyncio, time, random, contextlib, logging
from dataclasses import dataclass, field
from typing import AsyncIterator, Callable

logging.basicConfig(level=logging.INFO, format="%(asctime)s.%(msecs)03d | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("async-agent")

## 1. `asyncio` primitives — the 90-second refresher

- `async def` defines a **coroutine function**. Calling it returns a **coroutine object** — it does NOT run.
- `await` **suspends** the current coroutine and hands control back to the event loop.
- `asyncio.create_task(coro)` schedules a coroutine to run **concurrently** and returns a `Task`.
- `asyncio.gather(*coros)` runs many concurrently and waits for all of them.
- **Blocking calls (`time.sleep`, `requests.get`) freeze the whole loop** — that's the #1 async agent bug.

In [ ]:
async def hello(name: str, delay: float) -> str:
    await asyncio.sleep(delay)   # NON-blocking — yields to the loop
    return f"hello {name} (slept {delay}s)"

async def demo_primitives():
    t0 = time.perf_counter()
    # Sequential — the awaits are serialised
    a = await hello("A", 1.0)
    b = await hello("B", 1.0)
    print(f"sequential: {time.perf_counter()-t0:.2f}s  ->  {a} | {b}")

    t0 = time.perf_counter()
    # Concurrent — both waits overlap
    a, b = await asyncio.gather(hello("A", 1.0), hello("B", 1.0))
    print(f"gather:     {time.perf_counter()-t0:.2f}s  ->  {a} | {b}")

asyncio.run(demo_primitives())

## 2. Fake LLM and fake tools

We simulate an agent workload with realistic latencies so we can measure wall time without paying for tokens.
- `fake_llm(prompt)` — 2-4s think time
- `fake_llm_stream(prompt)` — yields tokens with jitter
- `web_search(q)` / `vector_search(q)` / `sql_query(q)` — 0.3-1.5s each

In [ ]:
async def fake_llm(prompt: str, jitter: tuple[float, float] = (2.0, 4.0)) -> str:
    await asyncio.sleep(random.uniform(*jitter))
    return f"LLM_ANSWER(prompt={prompt[:32]!r})"

async def fake_llm_stream(prompt: str, n_tokens: int = 20) -> AsyncIterator[str]:
    for i in range(n_tokens):
        await asyncio.sleep(random.uniform(0.05, 0.15))
        yield f"tok{i} "

async def web_search(q: str) -> str:
    await asyncio.sleep(random.uniform(0.4, 1.2)); return f"web[{q}]"

async def vector_search(q: str) -> str:
    await asyncio.sleep(random.uniform(0.3, 0.9)); return f"vec[{q}]"

async def sql_query(q: str) -> str:
    await asyncio.sleep(random.uniform(0.6, 1.5)); return f"sql[{q}]"

TOOLS: dict[str, Callable] = {"web": web_search, "vector": vector_search, "sql": sql_query}

## 3. Sync vs Async: the wall-time trap

A 5-step agent, each step ~3s. **Sync = ~15s**. **Async gather = ~max step**.

In [ ]:
async def sync_style_agent():
    """Awaits sequentially — 'sync-in-async' anti-pattern."""
    t0 = time.perf_counter()
    r1 = await fake_llm("plan")
    r2 = await web_search("databricks async")
    r3 = await vector_search("agent framework")
    r4 = await sql_query("SELECT * FROM users")
    r5 = await fake_llm("summarise")
    return time.perf_counter()-t0, [r1,r2,r3,r4,r5]

async def truly_async_agent():
    """LLM plan first, then fan out the 3 tools, then final LLM."""
    t0 = time.perf_counter()
    r1 = await fake_llm("plan")
    r2, r3, r4 = await asyncio.gather(
        web_search("databricks async"),
        vector_search("agent framework"),
        sql_query("SELECT * FROM users"),
    )
    r5 = await fake_llm("summarise")
    return time.perf_counter()-t0, [r1,r2,r3,r4,r5]

async def bench():
    s, _ = await sync_style_agent();   print(f"sync-style : {s:5.2f}s")
    a, _ = await truly_async_agent();  print(f"async fan-out: {a:5.2f}s  (speed-up ~{s/a:.1f}x)")

asyncio.run(bench())

## 4. Parallel tool fan-out — `gather` and `TaskGroup`

`asyncio.gather` is the classic. `asyncio.TaskGroup` (3.11+) is **strictly better** for agent orchestration because:
- Exceptions propagate naturally — one failed tool cancels the group
- No orphan tasks left running after an error
- Structured lifetime tied to the `async with` block

In [ ]:
async def fanout_with_gather(query: str):
    results = await asyncio.gather(*(tool(query) for tool in TOOLS.values()), return_exceptions=True)
    return dict(zip(TOOLS.keys(), results))

async def fanout_with_taskgroup(query: str):
    out = {}
    async with asyncio.TaskGroup() as tg:
        tasks = {name: tg.create_task(tool(query)) for name, tool in TOOLS.items()}
    # After the block, ALL tasks are done (or the group already raised)
    return {name: t.result() for name, t in tasks.items()}

async def demo_fanout():
    t0 = time.perf_counter(); print(await fanout_with_gather("vector db"),     f"({time.perf_counter()-t0:.2f}s)")
    t0 = time.perf_counter(); print(await fanout_with_taskgroup("vector db"),  f"({time.perf_counter()-t0:.2f}s)")

asyncio.run(demo_fanout())

### 4.1 What happens when a tool blows up?

TaskGroup cancels its siblings — no dangling coroutines.

In [ ]:
async def flaky_tool():
    await asyncio.sleep(0.3); raise RuntimeError("vector index timeout")

async def slow_tool():
    try:
        await asyncio.sleep(3.0); return "done"
    except asyncio.CancelledError:
        print("  slow_tool: got cancelled — cleaning up"); raise

async def demo_taskgroup_error():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(flaky_tool())
            tg.create_task(slow_tool())
    except* RuntimeError as eg:
        # 3.11 ExceptionGroup syntax
        for e in eg.exceptions: print("caught:", e)

asyncio.run(demo_taskgroup_error())

## 5. Streaming token generation

Streaming = start sending bytes to the user **before** the agent finishes thinking. Async iterators (`async for`) are the mechanism.

In [ ]:
async def stream_agent(prompt: str):
    print("assistant: ", end="", flush=True)
    async for tok in fake_llm_stream(prompt):
        print(tok, end="", flush=True)
    print()

asyncio.run(stream_agent("explain async agents"))

### 5.1 Streaming + mid-stream tool call

Realistic agent: stream tokens, detect a `TOOL:` marker, pause the stream, run the tool, resume with the result.

In [ ]:
async def scripted_stream():
    parts = ["Let me check that. ", "TOOL:web(databricks agent framework) ", "Based on the result, ", "here is the answer."]
    for p in parts:
        await asyncio.sleep(0.4); yield p

async def stream_with_tools():
    async for chunk in scripted_stream():
        if chunk.startswith("TOOL:"):
            call = chunk[5:].strip()
            tool_name, arg = call.split("(", 1); arg = arg.rstrip(") ")
            print(f"\n[executing tool {tool_name}({arg!r})]", flush=True)
            result = await TOOLS[tool_name](arg)
            print(f"[tool result: {result}]", flush=True)
        else:
            print(chunk, end="", flush=True)
    print()

asyncio.run(stream_with_tools())

## 6. Concurrency control: `Semaphore`, timeout, retry, cancellation

Unbounded `gather` will get you rate-limited in minutes. Cap it.

In [ ]:
async def rate_limited_llm(prompt: str, sem: asyncio.Semaphore) -> str:
    async with sem:                        # at most N concurrent
        return await fake_llm(prompt, jitter=(0.5, 1.0))

async def demo_semaphore():
    sem = asyncio.Semaphore(3)             # cap at 3 concurrent LLM calls
    t0 = time.perf_counter()
    outs = await asyncio.gather(*(rate_limited_llm(f"q{i}", sem) for i in range(12)))
    print(f"12 calls, cap=3, elapsed={time.perf_counter()-t0:.2f}s  (~4 waves)")

asyncio.run(demo_semaphore())

In [ ]:
async def slow_llm():
    await asyncio.sleep(10); return "never returned in prod"

async def demo_timeout():
    try:
        async with asyncio.timeout(1.5):   # 3.11+ context-manager form
            r = await slow_llm(); print(r)
    except TimeoutError:
        print("timed out at 1.5s — falling back")

asyncio.run(demo_timeout())

In [ ]:
# Retry with exponential backoff — tenacity's async API
from tenacity import AsyncRetrying, stop_after_attempt, wait_exponential, retry_if_exception_type

class RateLimit(Exception): pass

_attempts = 0
async def flaky_llm(prompt: str):
    global _attempts; _attempts += 1
    if _attempts < 3: raise RateLimit(f"429 (attempt {_attempts})")
    return f"ok on attempt {_attempts}"

async def demo_retry():
    async for attempt in AsyncRetrying(
        retry=retry_if_exception_type(RateLimit),
        wait=wait_exponential(multiplier=0.2, max=2),
        stop=stop_after_attempt(5),
        reraise=True,
    ):
        with attempt:
            print(await flaky_llm("hi"))

asyncio.run(demo_retry())

In [ ]:
# Cancellation semantics — user closes the tab mid-agent-run
async def agent_step():
    try:
        await asyncio.sleep(5); return "done"
    except asyncio.CancelledError:
        print("  step: cancelled — flushing partial trace, closing HTTP client")
        raise                                     # ALWAYS re-raise CancelledError

async def demo_cancel():
    task = asyncio.create_task(agent_step())
    await asyncio.sleep(0.5)
    task.cancel()
    with contextlib.suppress(asyncio.CancelledError):
        await task

asyncio.run(demo_cancel())

## 7. Multi-agent supervisor with parallel specialist dispatch

In [ ]:
@dataclass
class Specialist:
    name: str
    tool: Callable
    async def run(self, q: str) -> str:
        thought = await fake_llm(f"{self.name} plan for {q}", jitter=(0.5, 1.0))
        evidence = await self.tool(q)
        return f"[{self.name}] {thought} | {evidence}"

async def supervisor(query: str):
    specialists = [Specialist("WebAgent", web_search),
                   Specialist("RagAgent", vector_search),
                   Specialist("SqlAgent", sql_query)]
    t0 = time.perf_counter()
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(s.run(query)) for s in specialists]
    partials = [t.result() for t in tasks]
    synthesised = await fake_llm("synthesise: " + " \n".join(partials), jitter=(0.6, 1.0))
    print(f"supervisor elapsed: {time.perf_counter()-t0:.2f}s")
    for p in partials: print(" ", p)
    print("final:", synthesised)

asyncio.run(supervisor("how does Databricks Vector Search async work"))

## 8. Background fire-and-forget with `create_task`

Use case: log the trace to MLflow / write to a memory store — the user shouldn't wait for it.

In [ ]:
BACKGROUND: set[asyncio.Task] = set()   # hold refs so tasks aren't GC'd

async def write_trace(trace: dict):
    await asyncio.sleep(0.8)
    log.info("trace written: %s", trace)

def fire_and_forget(coro):
    t = asyncio.create_task(coro)
    BACKGROUND.add(t); t.add_done_callback(BACKGROUND.discard)
    return t

async def agent_with_bg_logging(q: str):
    answer = await fake_llm(q, jitter=(0.3, 0.6))
    fire_and_forget(write_trace({"q": q, "answer": answer}))
    return answer  # returns immediately, trace flushes in background

async def demo_bg():
    r = await agent_with_bg_logging("what is uvloop"); print("user got:", r)
    await asyncio.gather(*BACKGROUND)  # wait at shutdown so we don't lose logs

asyncio.run(demo_bg())

## 9. Wrapping a blocking library — the escape hatch

Sometimes you're stuck with a sync SDK (`psycopg2`, some vendor client). `asyncio.to_thread` offloads it to a threadpool so it doesn't freeze the loop.

In [ ]:
def blocking_sync_call(x: int) -> int:
    time.sleep(1.0)  # sync sleep — would kill the loop if awaited directly
    return x * x

async def demo_to_thread():
    t0 = time.perf_counter()
    outs = await asyncio.gather(*(asyncio.to_thread(blocking_sync_call, i) for i in range(5)))
    print(f"5 blocking calls, threaded gather: {time.perf_counter()-t0:.2f}s -> {outs}")

asyncio.run(demo_to_thread())

## 10. Real-provider snippets (READ-ONLY — gated on env vars)

These cells only execute if the corresponding API key / workspace is set. They show the exact async idiom you'll use in production.

In [ ]:
import os

async def openai_async_demo():
    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not set — skipping"); return
    from openai import AsyncOpenAI
    client = AsyncOpenAI()
    stream = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "one-line joke about event loops"}],
        stream=True,
    )
    async for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
    print()

asyncio.run(openai_async_demo())

In [ ]:
async def anthropic_async_demo():
    if not os.getenv("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY not set — skipping"); return
    from anthropic import AsyncAnthropic
    client = AsyncAnthropic()
    async with client.messages.stream(
        model="claude-3-5-sonnet-latest",
        max_tokens=200,
        messages=[{"role": "user", "content": "one-line joke about asyncio"}],
    ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)
    print()

asyncio.run(anthropic_async_demo())

In [ ]:
# Databricks Model Serving — async via httpx.AsyncClient.
# Env vars expected: DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_ENDPOINT
async def databricks_serving_demo():
    host = os.getenv("DATABRICKS_HOST"); tok = os.getenv("DATABRICKS_TOKEN"); ep = os.getenv("DATABRICKS_ENDPOINT")
    if not (host and tok and ep):
        print("Databricks env vars not set — skipping"); return
    import httpx
    url = f"{host}/serving-endpoints/{ep}/invocations"
    payload = {"messages": [{"role": "user", "content": "one line: what is uvloop"}], "max_tokens": 100}
    async with httpx.AsyncClient(timeout=30) as c:                       # always use async with
        r = await c.post(url, headers={"Authorization": f"Bearer {tok}"}, json=payload)
        r.raise_for_status(); print(r.json())

asyncio.run(databricks_serving_demo())

In [ ]:
# Databricks Vector Search — the SDK is sync; use asyncio.to_thread OR wrap in a small async facade.
async def dbx_vector_search_demo():
    if not os.getenv("DATABRICKS_HOST"):
        print("Databricks env vars not set — skipping"); return
    try:
        from databricks.vector_search.client import VectorSearchClient
    except ImportError:
        print("pip install databricks-vectorsearch to run this cell"); return
    index_name = os.getenv("DBX_VS_INDEX", "main.default.my_index")
    def _query():
        vsc = VectorSearchClient()
        idx = vsc.get_index(index_name=index_name)
        return idx.similarity_search(query_text="async agents", columns=["id","content"], num_results=3)
    result = await asyncio.to_thread(_query)
    print(result)

asyncio.run(dbx_vector_search_demo())

## 11. Load harness — how far does async take you?

Simulate N concurrent users hitting the agent, cap outbound LLM concurrency with a semaphore, and observe p50 / p95.

In [ ]:
import statistics

async def full_agent(q: str, sem: asyncio.Semaphore):
    t0 = time.perf_counter()
    async with sem:
        _ = await fake_llm(f"plan {q}", jitter=(0.4, 0.8))
        async with asyncio.TaskGroup() as tg:
            t_web = tg.create_task(web_search(q))
            t_vec = tg.create_task(vector_search(q))
            t_sql = tg.create_task(sql_query(q))
        _ = await fake_llm("synthesise", jitter=(0.4, 0.8))
    return time.perf_counter() - t0

async def load_test(n_users: int, concurrency_cap: int):
    sem = asyncio.Semaphore(concurrency_cap)
    t0 = time.perf_counter()
    lats = await asyncio.gather(*(full_agent(f"q{i}", sem) for i in range(n_users)))
    wall = time.perf_counter() - t0
    lats.sort()
    p50 = statistics.median(lats)
    p95 = lats[int(0.95 * len(lats)) - 1]
    print(f"users={n_users:3d} cap={concurrency_cap:2d} | wall={wall:5.2f}s | p50={p50:.2f}s p95={p95:.2f}s | throughput={n_users/wall:5.1f} rps")

async def sweep():
    for users, cap in [(10, 5), (25, 5), (25, 15), (50, 15)]:
        await load_test(users, cap)

asyncio.run(sweep())

## 12. Production checklist — mapped to cells above

| Check | Cell |
|-------|------|
| No blocking libs in hot path | §9 `to_thread` |
| LLM SDKs use async clients | §10 (OpenAI / Anthropic / Databricks) |
| Tool fan-out uses gather / TaskGroup | §4 |
| Concurrency cap via Semaphore | §6 |
| Every external call has a timeout | §6 `asyncio.timeout` |
| Retry with backoff on transient errors | §6 tenacity |
| Cancellation propagates + cleanup in `finally` | §6 `demo_cancel` |
| Streaming end-to-end | §5 |
| Background traces don't block response | §8 |
| Load-tested with concurrent users | §11 |

**Next steps**
- Wrap `full_agent` in FastAPI + uvicorn (uvloop) with `StreamingResponse`
- Point the fake tools at real Databricks Vector Search, UC Functions, and a Model Serving endpoint
- Add MLflow tracing (`mlflow.trace`) around each `async def` — spans nest naturally with the event loop

**References**
- [Databricks Agent Framework (Azure docs)](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/)
- [databrickslabs GitHub](https://github.com/databrickslabs)
- [PEP 3156 — asyncio](https://peps.python.org/pep-3156/)
- [Python 3.11 TaskGroup + `asyncio.timeout`](https://docs.python.org/3/library/asyncio-task.html#task-groups)